# 05 Feature table

One table for all models: a row per target half-hour and run cutoff (07:00 or 09:00 UK on D-1), development days only. Every feature is built from information available at that cutoff; the leak checks at the end verify this independently of the construction code. The table is saved to `outputs/` (git-ignored) so later notebooks read the same inputs.

## Init

In [1]:
import sys
sys.path[:0] = ["notebooks", "."]  # _common imports from repo root or notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from _common import load, to_uk, dev_only, UK_TZ

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 100)

plt.rcParams.update({"figure.figsize": (11, 4), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "figure.dpi": 100})

price = dev_only(load("imbalance_price")).set_index("period_start")["price_gbp_per_mwh"]
grid = price.index
loc = to_uk(grid.to_series())
day_of = loc.dt.tz_localize(None).dt.normalize()        # UK day D of each period, tz-naive midnight
dev_days = pd.DatetimeIndex(sorted(day_of.unique()))
print(len(grid), "periods,", len(dev_days), "development days")

32351 periods, 674 development days


## 1. Cutoffs and forecast snapshots

Same logic as notebook 03: for each target period, the latest vintage whose availability (`received_at`, else `issued_at`) is at or before D-1 HH:00 UK. This version also returns the availability timestamp of the vintage used, so the leak checks can test it. `shift_h` moves the cutoff and exists only so the leak check can show that a deliberately wrong cutoff is caught.

In [2]:
def cutoff_utc(day, hour):
    d1 = pd.Timestamp(day) - pd.Timedelta(days=1)
    return (d1 + pd.Timedelta(hours=hour)).tz_localize(UK_TZ, ambiguous=False, nonexistent="shift_forward").tz_convert("UTC")

CUT = {h: pd.Series({d: cutoff_utc(d, h) for d in dev_days}) for h in (7, 9)}

def pit_snapshot(df, hour, value_cols, shift_h=0, key="period_start"):
    df = df.copy()
    df["avail"] = df["received_at"].fillna(df["issued_at"])
    day = to_uk(df[key]).dt.tz_localize(None).dt.normalize()
    df["cutoff"] = day.map(CUT[hour]) + pd.Timedelta(hours=shift_h)
    ok = df[df["avail"] <= df["cutoff"]]
    ok = ok.sort_values([key, "avail"]).drop_duplicates(key, keep="last")
    ok["age_h"] = (ok["cutoff"] - ok["avail"]).dt.total_seconds() / 3600
    return ok.set_index(key)[value_cols + ["avail", "age_h"]]

raw = {n: dev_only(load(n)) for n in ["demand_forecast_elexon", "wind_forecast_elexon", "wind_solar_forecast_embedded"]}
SPEC = {"demand": ("demand_forecast_elexon", ["demand_mw"]),
        "wind": ("wind_forecast_elexon", ["wind_mw"]),
        "emb": ("wind_solar_forecast_embedded", ["embedded_wind_mw", "embedded_solar_mw"])}
snap = {h: {k: pit_snapshot(raw[t], h, c) for k, (t, c) in SPEC.items()} for h in (7, 9)}
print({h: {k: len(v) for k, v in s.items()} for h, s in snap.items()})

{7: {'demand': 6710, 'wind': 16102, 'emb': 32237}, 9: {'demand': 32142, 'wind': 16126, 'emb': 32237}}


## 2. Forecast features on the half-hour grid

The hourly Elexon wind value is assigned to both half-hours of its hour. Ages are in hours before the cutoff. The 07:00-to-09:00 revision exists only on the 09:00 rows (where both vintages are known); on 07:00 rows it is NaN.

In [3]:
def on_grid(s, floor=False):
    x = s.reindex(grid.floor("h") if floor else grid)
    x.index = grid
    return x

def forecast_block(hour):
    s = snap[hour]
    f = pd.DataFrame(index=grid)
    f["demand_mw"] = on_grid(s["demand"]["demand_mw"])
    f["wind_mw"] = on_grid(s["wind"]["wind_mw"], floor=True)
    f["emb_wind_mw"] = on_grid(s["emb"]["embedded_wind_mw"])
    f["emb_solar_mw"] = on_grid(s["emb"]["embedded_solar_mw"])
    f["wind_total_mw"] = f["wind_mw"] + f["emb_wind_mw"]
    f["demand_age_h"] = on_grid(s["demand"]["age_h"])
    f["wind_age_h"] = on_grid(s["wind"]["age_h"], floor=True)
    f["emb_age_h"] = on_grid(s["emb"]["age_h"])
    avail = pd.concat([on_grid(s["demand"]["avail"]), on_grid(s["wind"]["avail"], floor=True), on_grid(s["emb"]["avail"])], axis=1)
    f["_avail_max"] = avail.max(axis=1)
    return f

FB = {h: forecast_block(h) for h in (7, 9)}
for c in ["demand_mw", "wind_mw", "emb_wind_mw", "emb_solar_mw"]:
    FB[9][c.replace("_mw", "") + "_rev"] = FB[9][c] - FB[7][c]
    FB[7][c.replace("_mw", "") + "_rev"] = np.nan
print(FB[9].notna().mean().round(3).to_string())

demand_mw        0.994
wind_mw          0.997
emb_wind_mw      0.996
emb_solar_mw     0.996
wind_total_mw    0.993
demand_age_h     0.994
wind_age_h       0.997
emb_age_h        0.996
_avail_max       1.000
demand_rev       0.207
wind_rev         0.995
emb_wind_rev     0.996
emb_solar_rev    0.996


## 3. Calendar features

UK local hour as a fraction, weekday, weekend and bank-holiday flags, a one-harmonic seasonal pair from day of year (smooth, instead of 12 month dummies), and a flag for 46/50-period clock-change days. Bank holidays (England and Wales) are listed by hand to avoid a new dependency; they cover the data range.

In [4]:
BANK_HOLIDAYS = pd.to_datetime([
    "2024-12-25", "2024-12-26",
    "2025-01-01", "2025-04-18", "2025-04-21", "2025-05-05", "2025-05-26", "2025-08-25", "2025-12-25", "2025-12-26",
    "2026-01-01", "2026-04-03", "2026-04-06", "2026-05-04", "2026-05-25", "2026-08-31"])

cal = pd.DataFrame(index=grid)
cal["hour_frac"] = loc.dt.hour + loc.dt.minute / 60
cal["weekday"] = loc.dt.day_of_week
cal["is_weekend"] = (cal["weekday"] >= 5).astype(int)
cal["bank_holiday"] = day_of.isin(BANK_HOLIDAYS).astype(int).to_numpy()
doy = loc.dt.day_of_year
cal["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
cal["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)
n_per_day = day_of.map(day_of.value_counts())
cal["clock_change_day"] = (day_of.map(loc.groupby(day_of).apply(lambda s: s.dt.strftime("%z").nunique())) > 1).astype(int).to_numpy()
print(cal.describe().round(2).T.to_string())
print("clock-change days:", sorted(day_of[cal["clock_change_day"] == 1].dt.date.unique()))

                    count   mean   std  min   25%    50%    75%   max
hour_frac         32351.0  11.75  6.93  0.0  5.75  11.50  17.75  23.5
weekday           32351.0   3.00  2.00  0.0  1.00   3.00   5.00   6.0
is_weekend        32351.0   0.28  0.45  0.0  0.00   0.00   1.00   1.0
bank_holiday      32351.0   0.02  0.15  0.0  0.00   0.00   0.00   1.0
doy_sin           32351.0   0.07  0.69 -1.0 -0.57   0.12   0.75   1.0
doy_cos           32351.0   0.04  0.72 -1.0 -0.72   0.13   0.75   1.0
clock_change_day  32351.0   0.01  0.08  0.0  0.00   0.00   0.00   1.0
clock-change days: [datetime.date(2024, 10, 27), datetime.date(2025, 3, 30), datetime.date(2025, 10, 26), datetime.date(2026, 3, 29)]


## 4. Price-history features

All from prices known at the cutoff. Profile features use only days D-2 and earlier (full days, so safe at both cutoffs). Level features: `level24h` is the median of the 24 hours of prices ending 1 hour before the cutoff (the assumed publication delay from notebook 04), different at 07:00 and 09:00; `level7`/`level28` are medians over D-8..D-2 and D-29..D-2. `level_gap` = `level24h` - `level28` is the recent level relative to the longer-run level.

In [5]:
df = pd.DataFrame({"price": price, "day": day_of, "hhmm": loc.dt.strftime("%H:%M")})
df["occ"] = df.groupby(["day", "hhmm"]).cumcount()
wide = df.pivot(index="day", columns=["hhmm", "occ"], values="price")
wide = wide.reindex(pd.date_range(wide.index.min(), wide.index.max(), freq="D"))
known = wide.shift(2)                                  # row D shows the prices of D-2

def to_long(F):
    ri = F.index.get_indexer(df["day"])
    ci = F.columns.get_indexer(pd.MultiIndex.from_frame(df[["hhmm", "occ"]]))
    out = np.full(len(df), np.nan)
    ok = (ri >= 0) & (ci >= 0)
    out[ok] = F.to_numpy()[ri[ok], ci[ok]]
    return pd.Series(out, index=grid)

PH = pd.DataFrame(index=grid)
PH["d2_same"] = to_long(known)
PH["med7_same"] = to_long(known.rolling(7, min_periods=4).median())
PH["med28_same"] = to_long(known.rolling(28, min_periods=14).median())

vals = wide.to_numpy()
def flat_median(i, n):
    """median over the n full days D-n-1 .. D-2 for the day at row i"""
    lo, hi = max(0, i - 1 - n), max(0, i - 1)      # rows lo .. hi-1  (D-n-1 .. D-2); empty for the first row
    v = vals[lo:hi].ravel()
    return np.nanmedian(v) if np.isfinite(v).sum() >= 24 else np.nan

lvl7 = pd.Series({d: flat_median(i, 7) for i, d in enumerate(wide.index)})
lvl28 = pd.Series({d: flat_median(i, 28) for i, d in enumerate(wide.index)})
PH["level7"] = day_of.map(lvl7).to_numpy()
PH["level28"] = day_of.map(lvl28).to_numpy()

def level24(hour):
    t = price.index
    lv, last = {}, {}
    for d in wide.index:
        c = cutoff_utc(d, hour)
        m = (t >= c - pd.Timedelta(hours=25)) & (t <= c - pd.Timedelta(hours=1))
        w = price[m]
        lv[d] = w.median() if len(w) >= 24 else np.nan
        last[d] = w.index.max() if len(w) else pd.NaT
    return pd.Series(lv), pd.Series(last)

L24 = {h: level24(h) for h in (7, 9)}
print(PH.notna().mean().round(3).to_string())

d2_same       0.997
med7_same     0.992
med28_same    0.978
level7        0.997
level28       0.997


## 5. Daily margin features

One representative per correlated group from notebook 03/04 (the others are near-duplicates): minimum demand, national surplus, constrained plant, generator availability and interconnector import limit. The publication date is a midnight timestamp, so only publications dated D-2 or earlier are used (safe at either cutoff).

In [6]:
mg = dev_only(load("margin_daily"), col="issued_at").copy()
mg["day"] = pd.to_datetime(mg["day"])
mg["issued_day"] = to_uk(mg["issued_at"]).dt.tz_localize(None).dt.normalize()
mg = mg[(mg["day"] - mg["issued_day"]).dt.days >= 2].sort_values("issued_day").drop_duplicates("day", keep="last").set_index("day")
MARGIN_COLS = {"minimum_demand_forecast_mw": "mgn_min_demand", "national_surplus_mw": "mgn_surplus",
               "constrained_plant_mw": "mgn_constrained", "generator_availability_mw": "mgn_gen_avail",
               "max_interconnector_import_mw": "mgn_ic_import"}
MG = pd.DataFrame({new: day_of.map(mg[old]).to_numpy() for old, new in MARGIN_COLS.items()}, index=grid)
MG_issued = day_of.map(mg["issued_day"]).to_numpy()
print(MG.notna().mean().round(3).to_string())

mgn_min_demand     1.0
mgn_surplus        1.0
mgn_constrained    1.0
mgn_gen_avail      1.0
mgn_ic_import      1.0


## 6. Assemble

Stack the 07:00 and 09:00 versions into one table (`cutoff_hour` identifies them). The two targets: `price`, and `y_dev` = price minus `med28_same`. `med28_same` is kept as a column so predictions on the deviation target can be turned back into prices.

In [7]:
parts = []
for h in (7, 9):
    t = pd.concat([FB[h], cal, PH, MG], axis=1)
    t["level24h"] = day_of.map(L24[h][0]).to_numpy()
    t["level_gap"] = t["level24h"] - t["level28"]
    t["_level_last"] = day_of.map(L24[h][1]).to_numpy()
    t["_margin_issued"] = MG_issued
    t["cutoff_hour"] = h
    t["day"] = day_of.to_numpy()
    t["cutoff_utc"] = day_of.map(CUT[h]).to_numpy()
    t["price"] = price
    t["y_dev"] = t["price"] - t["med28_same"]
    parts.append(t)
T = pd.concat(parts).rename_axis("period_start").reset_index()
T = T.sort_values(["period_start", "cutoff_hour"]).reset_index(drop=True)
print(T.shape)
print(T.groupby("cutoff_hour").size().to_string())

(64702, 40)
cutoff_hour
7    32351
9    32351


## 6b. Anomaly features (forecast relative to its own recent typical value)

The demand, wind and solar forecasts as absolute levels carry seasonal information that the price reference already contains (notebooks 12-13). Each anomaly is the point-in-time forecast for the period minus the mean of the point-in-time forecasts for the same clock time on days D-29..D-2, at the same cutoff hour. Those earlier forecasts were each produced at their own cutoff (D'-1 at the same hour with D' <= D-2), so all were available at the cutoff of day D and have the same lead time as the forecast they are compared with. A clock time needs at least 14 earlier values.

In [8]:
T["_hhmm"] = to_uk(T["period_start"]).dt.strftime("%H:%M")
T["_occ"] = T.groupby(["cutoff_hour", "day", "_hhmm"]).cumcount()
ANOM_SRC = {"demand_mw": "demand_anom", "wind_mw": "wind_anom", "emb_wind_mw": "emb_wind_anom",
            "emb_solar_mw": "emb_solar_anom", "wind_total_mw": "wind_total_anom"}

def typical(sub, src):
    """28-day same-clock-time mean of `src` over days D-29..D-2, aligned to the rows of `sub`."""
    w = sub.pivot(index="day", columns=["_hhmm", "_occ"], values=src)
    w = w.reindex(pd.date_range(w.index.min(), w.index.max(), freq="D"))
    typ = w.shift(2).rolling(28, min_periods=14).mean()
    ri = typ.index.get_indexer(sub["day"])
    ci = typ.columns.get_indexer(pd.MultiIndex.from_frame(sub[["_hhmm", "_occ"]]))
    out = np.full(len(sub), np.nan)
    ok = (ri >= 0) & (ci >= 0)
    out[ok] = typ.to_numpy()[ri[ok], ci[ok]]
    return out

for h in (7, 9):
    sub = T[T["cutoff_hour"] == h]
    for src, new in ANOM_SRC.items():
        T.loc[sub.index, new] = sub[src].to_numpy() - typical(sub, src)
print(T[list(ANOM_SRC.values())].describe().round(0).T.to_string())
print(T.groupby("cutoff_hour")[list(ANOM_SRC.values())].apply(lambda g: g.isna().mean()).T.round(3).to_string())

                   count   mean     std      min     25%    50%     75%      max
demand_anom      37973.0 -107.0  2851.0 -19726.0 -1667.0   19.0  1574.0  11783.0
wind_anom        63006.0  -64.0  4966.0 -11640.0 -3664.0 -691.0  3617.0  14856.0
emb_wind_anom    63024.0  -13.0  1279.0  -2683.0  -947.0 -250.0   780.0   4608.0
emb_solar_anom   63024.0   69.0  1138.0  -8056.0   -24.0    0.0    68.0   6123.0
wind_total_anom  62776.0  -78.0  6187.0 -14170.0 -4551.0 -981.0  4394.0  19515.0
cutoff_hour          7      9
demand_anom      0.797  0.029
wind_anom        0.027  0.025
emb_wind_anom    0.026  0.026
emb_solar_anom   0.026  0.026
wind_total_anom  0.031  0.029


## 7. Leak checks

Independent of the construction code where possible. (a) Availability of every forecast vintage and margin publication used is at or before the cutoff. (b) The last price used in the level feature is at least 1 hour before the cutoff. (c) A deliberately wrong cutoff (24 h too late) is caught by the same test, which shows the test can fail. (d) A brute-force re-derivation of the demand/wind/embedded values and the price profiles for random rows matches the table.

In [9]:
# (a) availability <= cutoff
bad_fc = (T["_avail_max"] > T["cutoff_utc"]).sum()
mg_cut = T["cutoff_utc"]
bad_mg = (T["_margin_issued"].dt.tz_localize(UK_TZ) > mg_cut).sum()
print("(a) forecast vintages after cutoff:", bad_fc, "| margin publications after cutoff:", bad_mg)
assert bad_fc == 0 and bad_mg == 0

# (b) last price in the level window
gap = (T["cutoff_utc"] - T["_level_last"]).dt.total_seconds() / 3600
print("(b) hours between last price used and cutoff: min", round(gap.min(), 2), " (required >= 1)")
assert gap.min() >= 1 - 1e-9

# (c) the test is able to fail: build demand with a cutoff 24 h too late
wrong = pit_snapshot(raw["demand_forecast_elexon"], 9, ["demand_mw"], shift_h=24)
wrong_cut = wrong.index.to_series().pipe(lambda s: to_uk(s).dt.tz_localize(None).dt.normalize()).map(CUT[9])
share = (wrong["avail"] > wrong_cut.to_numpy()).mean()
print(f"(c) deliberately shifted cutoff: {share:.1%} of rows would use a vintage after the true cutoff (must be > 0)")
assert share > 0

(a) forecast vintages after cutoff: 0 | margin publications after cutoff: 0
(b) hours between last price used and cutoff: min 1.0  (required >= 1)


(c) deliberately shifted cutoff: 100.0% of rows would use a vintage after the true cutoff (must be > 0)


In [10]:
# (d) brute force on random rows: filter the raw table for the period directly
rng = np.random.default_rng(0)
sample = T.sample(120, random_state=1)
mism = 0
for _, r in sample.iterrows():
    ps, h, c = r["period_start"], int(r["cutoff_hour"]), r["cutoff_utc"]
    for tab, col, key, tcol in [("demand_forecast_elexon", "demand_mw", ps, "demand_mw"),
                                ("wind_forecast_elexon", "wind_mw", ps.floor("h"), "wind_mw"),
                                ("wind_solar_forecast_embedded", "embedded_solar_mw", ps, "emb_solar_mw")]:
        d = raw[tab]
        d = d[d["period_start"] == key]
        d = d.assign(a=d["received_at"].fillna(d["issued_at"]))
        d = d[d["a"] <= c].sort_values("a")
        exp = d[col].iloc[-1] if len(d) else np.nan
        got = r[tcol]
        if not ((np.isnan(exp) and np.isnan(got)) or exp == got):
            mism += 1
print("(d) forecast feature mismatches on 120 random rows x 3 tables:", mism)
assert mism == 0

# price profiles: direct computation from the raw price series
mism_p = 0
for _, r in T[T["cutoff_hour"] == 9].dropna(subset=["med28_same"]).sample(60, random_state=2).iterrows():
    D = r["day"]
    hh = to_uk(pd.Series([r["period_start"]])).dt.strftime("%H:%M").iloc[0]
    win = df[(df["day"] >= D - pd.Timedelta(days=29)) & (df["day"] <= D - pd.Timedelta(days=2)) & (df["hhmm"] == hh)]["price"]
    if len(win) >= 14 and abs(win.median() - r["med28_same"]) > 1e-9:
        mism_p += 1
    win2 = df[(df["day"] >= D - pd.Timedelta(days=29)) & (df["day"] <= D - pd.Timedelta(days=2))]["price"]
    if abs(win2.median() - r["level28"]) > 1e-9:
        mism_p += 1
print("(d) price-profile mismatches on 60 random rows:", mism_p)
assert mism_p == 0

(d) forecast feature mismatches on 120 random rows x 3 tables: 0
(d) price-profile mismatches on 60 random rows: 0


In [11]:
# (e) anomaly features: brute-force re-derivation from the table for random rows, and a check that the test can fail
def brute_anom(r, src, include_today=False):
    sub = T[(T["cutoff_hour"] == r["cutoff_hour"]) & (T["_hhmm"] == r["_hhmm"]) & (T["_occ"] == r["_occ"])]
    hi = r["day"] if include_today else r["day"] - pd.Timedelta(days=2)
    win = sub[(sub["day"] >= r["day"] - pd.Timedelta(days=29)) & (sub["day"] <= hi)][src].dropna()
    return r[src] - win.mean() if len(win) >= 14 else np.nan

mism_a, mism_leak, n_leak = 0, 0, 0
for _, r in T.sample(400, random_state=3).iterrows():
    for src, new in ANOM_SRC.items():
        exp = brute_anom(r, src)
        got = r[new]
        if not ((np.isnan(exp) and np.isnan(got)) or abs(exp - got) < 1e-6):
            mism_a += 1
        # the same computation with a window that includes day D (D-29..D) must disagree with the table
        wrong = brute_anom(r, src, include_today=True)
        if not np.isnan(wrong) and not np.isnan(got):
            n_leak += 1
            mism_leak += abs(wrong - got) > 1e-6
print("(e) anomaly mismatches vs brute force on 400 random rows x 5 features:", mism_a)
print(f"(e) window including day D would disagree with the table on {mism_leak} of {n_leak} comparisons (test can fail)")
assert mism_a == 0 and mism_leak > 0

# the window never reaches D-1 or D: the most recent day entering the rolling mean is D-2 by construction (shift(2));
# show it directly on one clock time
sub = T[(T["cutoff_hour"] == 9) & (T["_hhmm"] == "18:00") & (T["_occ"] == 0)].set_index("day")
d0 = sub.index[60]
manual = sub.loc[d0 - pd.Timedelta(days=29): d0 - pd.Timedelta(days=2), "demand_mw"].mean()
print("(e) example", d0.date(), "18:00: table typical =", round(sub.loc[d0, "demand_mw"] - sub.loc[d0, "demand_anom"], 2), " manual D-29..D-2 mean =", round(manual, 2))

(e) anomaly mismatches vs brute force on 400 random rows x 5 features: 0
(e) window including day D would disagree with the table on 1623 of 1784 comparisons (test can fail)
(e) example 2024-11-30 18:00: table typical = 38285.14  manual D-29..D-2 mean = 38285.14


## 8. Structure of the table

Rows per day (46/50 on clock-change days, one missing half-hour on 2026-01-18), missingness by cutoff, and correlations between the final features. The last check lists feature pairs with absolute correlation above 0.9.

In [12]:
rows = T.groupby(["cutoff_hour", "day"]).size().unstack(0)
print("rows per day by cutoff hour:"); print(rows.iloc[:, 0].value_counts().sort_index().to_string())
FEATURES = [c for c in T.columns if not c.startswith("_") and c not in
            ["period_start", "day", "cutoff_utc", "price", "y_dev", "cutoff_hour"]]
print(); print(len(FEATURES), "features")
print(T.groupby("cutoff_hour")[FEATURES + ["price"]].apply(lambda g: g.isna().mean()).T.round(3).to_string())

rows per day by cutoff hour:
7
46      2
47      1
48    669
50      2

36 features
cutoff_hour           7      9
demand_mw         0.793  0.006
wind_mw           0.005  0.003
emb_wind_mw       0.004  0.004
emb_solar_mw      0.004  0.004
wind_total_mw     0.008  0.007
demand_age_h      0.793  0.006
wind_age_h        0.005  0.003
emb_age_h         0.004  0.004
demand_rev        1.000  0.793
wind_rev          1.000  0.005
emb_wind_rev      1.000  0.004
emb_solar_rev     1.000  0.004
hour_frac         0.000  0.000
weekday           0.000  0.000
is_weekend        0.000  0.000
bank_holiday      0.000  0.000
doy_sin           0.000  0.000
doy_cos           0.000  0.000
clock_change_day  0.000  0.000
d2_same           0.003  0.003
med7_same         0.008  0.008
med28_same        0.022  0.022
level7            0.003  0.003
level28           0.003  0.003
mgn_min_demand    0.000  0.000
mgn_surplus       0.000  0.000
mgn_constrained   0.000  0.000
mgn_gen_avail     0.000  0.000
mgn_ic_import    

In [13]:
C = T[FEATURES].corr()
pairs = [(a, b, C.loc[a, b]) for i, a in enumerate(C.columns) for b in C.columns[i + 1:] if abs(C.loc[a, b]) > 0.9]
print("pairs with |corr| > 0.9:")
for a, b, v in sorted(pairs, key=lambda x: -abs(x[2])):
    print(f"  {a:18s} {b:18s} {v:+.3f}")

pairs with |corr| > 0.9:
  wind_mw            wind_total_mw      +0.997
  wind_anom          wind_total_anom    +0.997
  emb_wind_anom      wind_total_anom    +0.960
  emb_wind_mw        wind_total_mw      +0.959
  wind_anom          emb_wind_anom      +0.937
  wind_mw            emb_wind_mw        +0.937
  emb_wind_mw        emb_wind_anom      +0.913


## 9. Save

Saved for the modelling notebooks. `outputs/` is git-ignored; this notebook regenerates the file.

In [14]:
out = Path("outputs"); out.mkdir(exist_ok=True)
save_cols = ["period_start", "day", "cutoff_hour"] + FEATURES + ["price", "y_dev"]
T[save_cols].to_parquet(out / "feature_table_dev.parquet", index=False)
print(out / "feature_table_dev.parquet", T[save_cols].shape)
print(T[save_cols].tail(3).T.to_string())

outputs\feature_table_dev.parquet (64702, 41)
                                      64699                      64700                      64701
period_start      2026-08-05 22:00:00+00:00  2026-08-05 22:30:00+00:00  2026-08-05 22:30:00+00:00
day                     2026-08-05 00:00:00        2026-08-05 00:00:00        2026-08-05 00:00:00
cutoff_hour                               9                          7                          9
demand_mw                           21768.0                        NaN                    20390.0
wind_mw                             17441.0                    17441.0                    17441.0
emb_wind_mw                          3443.0                     3451.0                     3396.0
emb_solar_mw                            0.0                        0.0                        0.0
wind_total_mw                       20884.0                    20892.0                    20837.0
demand_age_h                        0.23629                        NaN  

## Findings

- **Table:** 64,702 rows (32,351 target periods x 2 cutoffs), 36 features (31 plus 5 anomaly features added in 6b) plus `price` and `y_dev`, development days only. Rows per day are 48 (669 days), 46 (2), 50 (2) and 47 (1, the missing half-hour on 2026-01-18), matching the clock changes. Saved to `outputs/feature_table_dev.parquet` (git-ignored).
- **Leak checks passed.** (a) No forecast vintage or margin publication used is after its cutoff. (b) The last price used in `level24h` is at least 1 hour before the cutoff. (c) Building demand with a cutoff 24 hours late puts every row after the true cutoff, so the test can fail. (d) A brute-force re-derivation from the raw tables matched on 120 random rows for each of three forecast tables, and `med28_same` / `level28` matched on 60 random rows.
- **Missingness.** At 07:00 demand is missing for 79.3% of rows (and so are its age and revision); at 09:00 it is 0.6%. All other forecast features are missing for 0.3-0.8%. `med28_same` is missing for 2.2% (the first 28 days), `d2_same` for 0.3%. Margin features have no gaps. The 07:00 revision features are all NaN by construction.
- **Redundant features.** Pairs above 0.9 correlation: `wind_mw`/`wind_total_mw` 0.997, `emb_wind_mw`/`wind_total_mw` 0.959, `wind_mw`/`emb_wind_mw` 0.937. For linear models use either `wind_total_mw` alone or the two parts, not all three. Margin duplicates were already dropped.
- **Calendar and level features** show no correlation above 0.9 with each other. `level24h` differs between the two cutoffs (for example 119.3 at 07:00 versus 113.5 at 09:00 on 2026-08-05 22:30 rows), so it is the only price feature that differs by cutoff.
- **Not yet done:** feature scaling, imputation (to be done inside folds for models that need it), the fold definitions, and any check of whether a feature adds information beyond the profile. Those belong to the modelling step.
- **Anomaly features (6b)**, added after the first modelling notebooks: demand, Elexon wind, embedded wind, embedded solar and total wind, each minus the mean of the same clock time's point-in-time forecasts over D-29..D-2 at the same cutoff hour. Missing for the first ~29 days (2.6-3.1%) and, for demand, wherever demand is missing (79.7% at 07:00, 2.9% at 09:00). Leak check (e): matches a brute-force re-derivation on 400 random rows x 5 features with 0 mismatches, and a window that includes day D disagrees with the table on 1,623 of 1,784 comparisons, so the test can fail. They correlate 0.91-0.997 with each other and with the absolute wind features, like the absolute versions.
- **Correction (found when the pipeline was ported to `sip_forecaster/`, notebook 20):** an earlier version of the `level7`/`level28` code used a slice that, for the first development day (2024-10-01) only, selected almost the whole price series, i.e. later prices. It is fixed above (empty window, so NaN). No model row was affected: that day has no `med28_same` (it needs 14 days of history) and the models train and score only on rows that have it.
